<a href="https://colab.research.google.com/github/aadiithi/tikitikitaka/blob/main/resnettrain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")

print("Path to dataset files:", path)



100%|██████████| 105M/105M [00:00<00:00, 200MB/s] 

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images/versions/3


In [ ]:
import numpy as np
import keras
from keras import layers
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt


ERROR:absl:Detected incompatible Protobuf Gencode/Runtime versions when loading tensorflow_metadata/proto/v0/anomalies.proto: gencode 6.31.1 runtime 5.29.6. Runtime version cannot be older than the linked gencode version. See Protobuf version guarantees at https://protobuf.dev/support/cross-version-runtime-guarantee.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/__init__.py", line 79, in <module>
    from tensorflow_datasets import rlds  # pylint: disable=g-bad-import-order
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/rlds/__init__.py", line 21, in <module>
    from tensorflow_datasets.rlds import envlogger_reader
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/rlds/envlogger_reader.py", line 21, in <module>
    from tensorflow_datasets.core.utils.lazy_imports_utils import tree
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/co

In [ ]:
import keras

# This part sets up the feature extraction:
# Load the pre-trained ResNet50 model from Keras applications.
# We include 'include_top=False' to remove the original classification layer,
# allowing the model to act as a fixed feature extractor.
# The input_shape is set for image data expected by ResNet50.
resnet_model = keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Print a summary of the model to see its architecture.
# Notice the absence of the final classification layers.
resnet_model.summary()


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "resnet50"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 23,587,712 (89.98 MB)

 Trainable params: 23,534,592 (89.78 MB)

 Non-trainable params: 53,120 (207.50 KB)

In [ ]:
layer = keras.layers.BatchNormalization()
layer.build((None, 4))  # Create the weights

print("weights:", len(layer.weights))
print("trainable_weights:", len(layer.trainable_weights))
print("non_trainable_weights:", len(layer.non_trainable_weights))


weights: 4
trainable_weights: 2
non_trainable_weights: 2


In [ ]:
layer = keras.layers.Dense(3)
layer.build((None, 4))  # Create the weights
layer.trainable = False  # Freeze the layer

print("weights:", len(layer.weights))
print("trainable_weights:", len(layer.trainable_weights))
print("non_trainable_weights:", len(layer.non_trainable_weights))


weights: 2
trainable_weights: 0
non_trainable_weights: 2


In [ ]:
import tensorflow as tf

# This creates a GlobalAveragePooling2D layer, which reduces the spatial dimensions
# of the features extracted by ResNet50, preparing them for the dense classification layers.
x = tf.keras.layers.GlobalAveragePooling2D()(resnet_model.output)

# This is the new classification head for your specific task.
# A Dense layer with 'relu' activation for hidden representation.
x = tf.keras.layers.Dense(256, activation='relu')(x)
# A Dropout layer for regularization to prevent overfitting.
x = tf.keras.layers.Dropout(0.5)(x)
# The final output Dense layer. Assuming 2 classes (real/fake), so 1 unit with sigmoid for binary classification.
# If you have more than 2 classes, you'd use units=num_classes and 'softmax' activation.
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

# Combine the ResNet50 base (feature extractor) with the new classification head.
model = tf.keras.Model(inputs=resnet_model.input, outputs=outputs)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 24,112,513 (91.98 MB)

 Trainable params: 24,059,393 (91.78 MB)

 Non-trainable params: 53,120 (207.50 KB)

### Dataset Preprocessing

We need to prepare the CIFake dataset for input into the ResNet50 model. This involves several steps:

1.  **Loading Data**: Using `tf.keras.utils.image_dataset_from_directory` to efficiently load images from the specified directory structure.
2.  **Image Resizing**: ResNet50 expects input images of size 224x224 pixels. We will resize all images to this dimension.
3.  **Pixel Rescaling**: Normalize pixel values from the original `[0, 255]` range to `[0, 1]`. This helps the model converge faster and perform better.
4.  **Splitting Data**: Divide the dataset into training, validation, and testing sets.
5.  **Batching and Prefetching**: Optimize data loading for training efficiency by batching images and prefetching them.

In [ ]:
# Define image dimensions expected by ResNet50
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32

# Construct the full paths to the real and fake image directories
cifake_path = path # 'path' comes from the kagglehub.dataset_download cell
train_dir = tf.keras.utils.image_dataset_from_directory(
    cifake_path, # Assuming the dataset has 'train' and 'test' subdirectories
    labels='inferred',
    label_mode='binary',
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    interpolation='nearest',
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42,
    validation_split=0.2, # Use 20% of the training data for validation
    subset='training'
)

val_dir = tf.keras.utils.image_dataset_from_directory(
    cifake_path,
    labels='inferred',
    label_mode='binary',
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    interpolation='nearest',
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=42,
    validation_split=0.2,
    subset='validation'
)

# Assuming a separate 'test' folder if not using validation_split for test
# For simplicity, if a separate test set is not available, you might split validation further.
# Given the dataset structure is usually organized, we'll assume a direct load is possible.
# If cifake_path itself is the root for train/test folders:
# For now, let's just use the train/validation split from the main directory for simplicity.
# If you have a separate test set, you would load it similarly:
# test_dir = tf.keras.utils.image_dataset_from_directory(test_data_path, ...)

# To have a test set, we will take a portion of the validation set
val_batches = tf.data.experimental.cardinality(val_dir)
test_dir = val_dir.take(val_batches // 2)
val_dir = val_dir.skip(val_batches // 2)

# Function to rescale pixel values to [0, 1]
def preprocess(image, label):
    # Fix: Cast image to float32 BEFORE division
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

# Apply preprocessing to all datasets
train_dir = train_dir.map(preprocess).cache().prefetch(buffer_size=tf.data.AUTOTUNE)
val_dir = val_dir.map(preprocess).cache().prefetch(buffer_size=tf.data.AUTOTUNE)
test_dir = test_dir.map(preprocess).cache().prefetch(buffer_size=tf.data.AUTOTUNE)

print(f"Number of training batches: {tf.data.experimental.cardinality(train_dir)}")
print(f"Number of validation batches: {tf.data.experimental.cardinality(val_dir)}")
print(f"Number of test batches: {tf.data.experimental.cardinality(test_dir)}")


Found 120000 files belonging to 2 classes.
Using 96000 files for training.
Found 120000 files belonging to 2 classes.
Using 24000 files for validation.
Number of training batches: 3000
Number of validation batches: 375
Number of test batches: 375


### Phase 1: Train the New Classification Head (Feature Extraction)

In this initial phase, we will keep the entire pre-trained `resnet_model` frozen and only train the newly added classification layers (`GlobalAveragePooling2D`, `Dense`, `Dropout`, final `Dense`). This allows the model to quickly learn to classify your specific data based on the powerful features already extracted by ResNet50, without altering its pre-trained weights.

In [ ]:
# Freeze the ResNet50 base model (feature extractor) so its weights are not updated during this phase.
resnet_model.trainable = False

# Compile the model with an optimizer, loss function, and metrics.
# We use binary_crossentropy for binary classification and Adam optimizer.
# metrics=['accuracy'] will track the accuracy during training.
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), # Standard learning rate for initial training
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

model.summary()

# Train the model for a few epochs.
# This will only update the weights of the newly added classification layers.
initial_epochs = 10 # You might adjust this based on your dataset size and convergence.

history = model.fit(
    train_dir,
    epochs=initial_epochs,
    validation_data=val_dir
)


NameError: name 'resnet_model' is not defined

### Phase 2: Fine-tuning (Unfreeze and Train Last Layers of ResNet50)

Now that the new classification head has learned to interpret the high-level features, we can unfreeze a portion of the `resnet_model` (typically the last few convolutional blocks) and continue training the entire model with a very small learning rate. This allows the model to slightly adjust the pre-trained weights of ResNet50 to better suit the specific characteristics of your CIFake dataset, potentially improving performance further.

In [ ]:
# Unfreeze the ResNet50 base model.
resnet_model.trainable = True

# It's important to only unfreeze a portion of the base model.
# Let's unfreeze the last few layers/blocks of ResNet50.
# You might need to experiment with how many layers to unfreeze.
# For example, to unfreeze the last 20 layers:
for layer in resnet_model.layers[:-20]:
    layer.trainable = False

# Recompile the model with a much lower learning rate for fine-tuning.
# A very small learning rate prevents large changes to the pre-trained weights.
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), # Very low learning rate for fine-tuning
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

model.summary()

# Continue training the model for more epochs with the unfrozen layers.
fine_tune_epochs = 10 # Additional epochs for fine-tuning
total_epochs = initial_epochs + fine_tune_epochs

history_fine = model.fit(
    train_dir,
    epochs=total_epochs,
    initial_epoch=history.epoch[-1],
    validation_data=val_dir
)


### Evaluate the Model and Predict Confidence Scores

Finally, let's evaluate the performance of our fine-tuned model on the test set and then make predictions on a few sample test images, outputting their confidence scores in JSON format.

In [ ]:
import json

# Evaluate the model on the test dataset
loss, accuracy = model.evaluate(test_dir)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# Get a few test images and their true labels
sample_images = []
sample_labels = []

# Iterate through a few batches of the test dataset
for images, labels in test_dir.take(2): # Take 2 batches for sampling
    sample_images.append(images)
    sample_labels.append(labels)

sample_images = tf.concat(sample_images, axis=0)
sample_labels = tf.concat(sample_labels, axis=0)

# Make predictions on the sample images
predictions = model.predict(sample_images)

# Prepare output in JSON format
output_predictions = []
for i in range(len(sample_labels)): # Iterate through each sample image
    output_predictions.append({
        "image_index": i,
        "predicted_confidence": float(predictions[i][0]), # Sigmoid output is confidence for class 1
        "true_label": int(sample_labels[i].numpy()) # Convert tensor label to int
    })

# Save the output as JSON
json_output = json.dumps(output_predictions, indent=4)
print("\n--- Sample Predictions (JSON) ---")
print(json_output)


### Evaluate the Model and Predict Confidence Scores

Finally, let's evaluate the performance of our fine-tuned model on the test set and then make predictions on a few sample test images, outputting their confidence scores in JSON format.

In [ ]:
import json

# Evaluate the model on the test dataset
loss, accuracy = model.evaluate(test_dir)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# Get a few test images and their true labels
sample_images = []
sample_labels = []

# Iterate through a few batches of the test dataset
for images, labels in test_dir.take(2): # Take 2 batches for sampling
    sample_images.append(images)
    sample_labels.append(labels)

sample_images = tf.concat(sample_images, axis=0)
sample_labels = tf.concat(sample_labels, axis=0)

# Make predictions on the sample images
predictions = model.predict(sample_images)

# Prepare output in JSON format
output_predictions = []
for i in range(len(sample_labels)): # Iterate through each sample image
    output_predictions.append({
        "image_index": i,
        "predicted_confidence": float(predictions[i][0]), # Sigmoid output is confidence for class 1
        "true_label": int(sample_labels[i].numpy()) # Convert tensor label to int
    })

# Save the output as JSON
json_output = json.dumps(output_predictions, indent=4)
print("\n--- Sample Predictions (JSON) ---")
print(json_output)
